Entry
The bot opens a Long position when all conditions align for a powerful breakout:
MA9 crosses UP over MA50: The short-term price has surged past the medium-term average, signaling a fresh uptrend.
RSI is between 55 and 70: This is the strategy's "secret weapon." An RSI above 55 confirms genuine bullish momentum, but keeping it strictly below 70 ensures we are not buying at the top of an exhausted, overbought FOMO spike.
Exit & Risk Management
Unlike rigid strategies, this bot adapts to the market to protect your capital and maximize your wins:
Stop Loss: MA50 - (1.5 x ATR)
Instead of a fixed percentage, the bot calculates a dynamic stop loss placed just below the MA50, combined with a volatility buffer. This gives the trade room to breathe while keeping maximum risk tightly controlled.
Take Profit: MA9 crosses DOWN over MA50
There is no fixed Take Profit target. The bot follows the golden rule of trading: "Let your winners run." It will hold the position for as long as it lasts, only closing when the trend officially breaks down.


# Python Code

In [ ]:
import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# ==========================================
# GLOBAL CONFIGURATION
# ==========================================
CONFIG = {
    'SYMBOLS': ['SOLUSDT'],
    'INTERVAL': '1h',
    'START_DATE': '01/01/2023',
    'END_DATE': '01/01/2024',
    'INITIAL_CAPITAL': 10000.0,
    'POSITION_SIZE_PCT': 0.10, # Sử dụng 10% vốn mỗi lệnh
    'FEE_RATE': 0.0004,        # Phí giao dịch Binance VIP 0
    'SLIPPAGE': 0.0002         # Trượt giá
}

# ==========================================
# MODULE 1: DATA HANDLER
# ==========================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        req_symbol = self.symbol

        print(f"Downloading {req_symbol} ({self.interval})...")
        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451:
                base_url = "https://api.binance.us"
                if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
                if req_symbol == "BTCUSDT": req_symbol = "BTCUSD"
                if req_symbol == "SOLUSDT": req_symbol = "SOLUSD"
        except Exception:
            base_url = "https://api.binance.us"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)

        if self.end_date == datetime.now().strftime("%d/%m/%Y"):
            end_time = int(time.time() * 1000)
        else:
            end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
            end_time = int(end_dt.timestamp() * 1000)

        all_klines =[]

        while start_time < end_time:
            params = {"symbol": req_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df.astype(float)
        return df

# ==========================================
# MODULE 2: STRATEGY LOGIC
# ==========================================
class MaximizedMACrossingStrategy:
    def __init__(self):
        self.ma_fast = 9
        self.ma_slow = 50
        self.rsi_len = 14
        self.atr_len = 14
        self.atr_mult = 1.5

    def calculate_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        # 1. Moving Averages
        df['MA9'] = ta.sma(df['Close'], length=self.ma_fast)
        df['MA50'] = ta.sma(df['Close'], length=self.ma_slow)

        # 2. RSI
        df['RSI'] = ta.rsi(df['Close'], length=self.rsi_len)

        # 3. ATR
        df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=self.atr_len)

        df.dropna(inplace=True)
        return df

# ==========================================
# MODULE 3: BACKTEST ENGINE (LONG ONLY + SL ĐỘNG)
# ==========================================
class BacktestEngineLong:
    def __init__(self, df, strategy, config):
        self.df = strategy.calculate_indicators(df.copy())
        self.config = config

        self.capital = config['INITIAL_CAPITAL']
        self.invested_amount = 0.0
        self.position = 0 # 0: Trống, 1: LONG
        self.entry_price = 0.0
        self.stop_loss_price = 0.0 # Lưu trữ mức giá SL
        self.trades = []

        self.equity_curve = np.zeros(len(self.df))

    def run(self):
        O = self.df['Open'].values
        H = self.df['High'].values
        L = self.df['Low'].values
        C = self.df['Close'].values

        ma9 = self.df['MA9'].values
        ma50 = self.df['MA50'].values
        rsi = self.df['RSI'].values
        atr = self.df['ATR'].values
        times = self.df.index

        fee = self.config['FEE_RATE']
        slip = self.config['SLIPPAGE']
        pos_size_pct = self.config['POSITION_SIZE_PCT']

        self.equity_curve = np.full(len(self.df), self.capital)

        for i in range(2, len(self.df)): # Bắt đầu từ 2 để check giao cắt (i-1 và i-2)

            # --- 1. EXIT LOGIC (Đang có lệnh LONG) ---
            if self.position == 1:
                exit_signal = False
                reason = ""
                exit_price = 0.0

                # Kiểm tra Stop Loss (Giá Low chạm SL)
                if L[i] <= self.stop_loss_price:
                    exit_signal = True
                    reason = "Stop Loss Hit"
                    # Nếu giá mở cửa đã thủng SL (Gap down), phải khớp ở giá Open. Nếu không, khớp ở giá SL.
                    exit_price = min(self.stop_loss_price, O[i])

                # Kiểm tra Take Profit (MA9 cắt XUỐNG MA50) ở nến trước đó
                elif (ma9[i-1] < ma50[i-1]) and (ma9[i-2] >= ma50[i-2]):
                    exit_signal = True
                    reason = "MA Death Cross (TP)"
                    exit_price = O[i]

                # Thực thi Exit
                if exit_signal:
                    real_exit = exit_price * (1 - slip) # Long Exit = Bán (Trừ đi slippage)

                    pnl_pct = (real_exit - self.entry_price) / self.entry_price
                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = self.invested_amount * net_pnl_pct

                    self.capital += pnl_amt

                    self.trades.append({
                        'Date': times[i], 'Type': 'Long', 'Reason': reason,
                        'Entry': self.entry_price, 'Exit': real_exit,
                        'PnL_Amt': pnl_amt, 'Equity': self.capital
                    })

                    self.position = 0
                    self.invested_amount = 0.0
                    self.stop_loss_price = 0.0

            # --- 2. ENTRY LOGIC (Chưa có lệnh) ---
            elif self.position == 0:
                # Điều kiện: MA9 cắt LÊN MA50
                cross_up = (ma9[i-1] > ma50[i-1]) and (ma9[i-2] <= ma50[i-2])

                # Điều kiện: 55 < RSI < 70
                rsi_condition = (55 < rsi[i-1] < 70)

                if cross_up and rsi_condition:
                    self.position = 1
                    self.entry_price = O[i] * (1 + slip) # Long Entry = Mua (Cộng slippage)
                    self.invested_amount = self.capital * pos_size_pct

                    # Tính toán Stop Loss: MA50 - (1.5 * ATR) của nến tín hiệu
                    self.stop_loss_price = ma50[i-1] - (1.5 * atr[i-1])

            # --- 3. UNREALIZED PNL ---
            current_equity = self.capital
            if self.position == 1:
                unrealized_pnl_pct = (C[i] - self.entry_price) / self.entry_price
                unrealized_pnl = self.invested_amount * unrealized_pnl_pct
                current_equity += unrealized_pnl

            self.equity_curve[i] = current_equity

        return pd.Series(self.equity_curve, index=self.df.index), pd.DataFrame(self.trades)

# ==========================================
# MODULE 4: PERFORMANCE EVALUATOR (Giữ nguyên)
# ==========================================
class Evaluator:
    @staticmethod
    def generate_report(equity, trades_df, initial_capital, symbol, interval, pos_size_pct):
        if trades_df.empty:
            print(f"No trades executed for {symbol}!")
            return

        final_equity = equity.iloc[-1]
        net_pnl = final_equity - initial_capital
        total_return = net_pnl / initial_capital

        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        max_dd = drawdown.min()

        wins = trades_df[trades_df['PnL_Amt'] > 0]
        losses = trades_df[trades_df['PnL_Amt'] <= 0]
        winrate = len(wins) / len(trades_df) if len(trades_df) > 0 else 0
        profit_factor = wins['PnL_Amt'].sum() / abs(losses['PnL_Amt'].sum()) if len(losses) > 0 and losses['PnL_Amt'].sum() != 0 else 0

        days = (equity.index[-1] - equity.index[0]).days
        cagr = ((final_equity / initial_capital) ** (365.0 / days)) - 1 if days > 0 else 0

        def get_annual_factor(tf):
            if tf == '15m': return 365 * 24 * 4
            elif tf == '1h': return 365 * 24
            elif tf == '4h': return 365 * 6
            elif tf == '1d': return 365
            return 365 * 24

        annual_factor = get_annual_factor(interval)
        returns = equity.pct_change().dropna()
        sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(annual_factor) if returns.std() != 0 else 0

        print("")
        print(f"========== PERFORMANCE REPORT: {symbol} ({interval}) ==========")
        print(f"{'Strategy:':>20} Maximized MA Crossing")
        print(f"{'Position Sizing:':>20} {pos_size_pct*100}% of Capital")
        print(f"{'Initial Capital:':>20} ${initial_capital:,.2f}")
        print(f"{'Final Equity:':>20} ${final_equity:,.2f}")
        print(f"{'Net PnL:':>20} ${net_pnl:,.2f}")
        print(f"{'Total Return:':>20} {total_return:.2%}")
        print(f"{'CAGR (Annual):':>20} {cagr:.2%}")
        print(f"{'Max Drawdown:':>20} {max_dd:.2%}")
        print(f"{'Sharpe Ratio:':>20} {sharpe_ratio:.2f}")
        print(f"{'Total Trades:':>20} {len(trades_df)}")
        print(f"{'Winrate:':>20} {winrate:.2%}")
        print(f"{'Profit Factor:':>20} {profit_factor:.2f}")
        print("====================================================\n")

        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.08,
            row_heights=[0.7, 0.3]
        )

        fig.add_trace(go.Scatter(
            x=equity.index, y=equity.values,
            mode="lines", name="Equity",
            line=dict(color="#2962ff", width=1.5)
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=drawdown.index, y=drawdown.values,
            mode="lines", name="Drawdown",
            line=dict(color="#d32f2f", width=1),
            fill='tozeroy', fillcolor='rgba(211, 47, 47, 0.2)'
        ), row=2, col=1)

        fig.update_layout(
            title=f"Portfolio Performance: {symbol} ({interval})",
            template="plotly_white",
            height=650,
            margin=dict(l=40, r=40, t=60, b=40),
            showlegend=False
        )

        fig.update_xaxes(showgrid=True, gridcolor='#e6e6e6', zeroline=False)
        fig.update_yaxes(title_text="Balance ($)", tickprefix="$", showgrid=True, gridcolor='#e6e6e6', zeroline=False, row=1, col=1)
        fig.update_yaxes(title_text="Drawdown %", tickformat=".1%", showgrid=True, gridcolor='#e6e6e6', zeroline=True, zerolinecolor='#cccccc', row=2, col=1)

        fig.show()

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    for symbol in CONFIG['SYMBOLS']:
        print(f"\n---> Starting Backtest for: {symbol} <---")
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            # Khởi tạo chiến lược bạn vừa cung cấp
            strategy = MaximizedMACrossingStrategy()

            # Sử dụng Engine được tinh chỉnh cho Long
            engine = BacktestEngineLong(df_raw, strategy, CONFIG)
            equity, trades = engine.run()

            Evaluator.generate_report(
                equity, trades,
                CONFIG['INITIAL_CAPITAL'], symbol,
                CONFIG['INTERVAL'], CONFIG['POSITION_SIZE_PCT']
            )

            # In ra 5 trade gần nhất để bạn kiểm chứng (Tùy chọn)
            if not trades.empty:
                print(f"--- Top 5 Recent Trades for {symbol} ---")
                print(trades[['Date', 'Reason', 'Entry', 'Exit', 'PnL_Amt']].tail())


---> Starting Backtest for: SOLUSDT <---

========== PERFORMANCE REPORT: SOLUSDT (1h) ==========
           Strategy: Maximized MA Crossing
    Position Sizing: 10.0% of Capital
    Initial Capital: $10,000.00
       Final Equity: $12,002.89
            Net PnL: $2,002.89
       Total Return: 20.03%
      CAGR (Annual): 20.15%
       Max Drawdown: -3.91%
       Sharpe Ratio: 2.58
       Total Trades: 84
            Winrate: 34.52%
      Profit Factor: 2.28



--- Top 5 Recent Trades for SOLUSDT ---
                  Date               Reason       Entry        Exit  \
79 2023-12-12 14:00:00        Stop Loss Hit   71.764350   69.016338   
80 2023-12-16 06:00:00  MA Death Cross (TP)   71.874372   74.385120   
81 2023-12-16 20:00:00        Stop Loss Hit   77.175432   73.094131   
82 2023-12-26 13:00:00  MA Death Cross (TP)   74.934984  116.666662   
83 2023-12-30 08:00:00        Stop Loss Hit  108.911778  101.323217   

       PnL_Amt  
79  -44.778359  
80   38.944384  
81  -61.460683  
82  633.251728  
83  -84.716064  


# Go Code
